# 📧 Analisis & Klasifikasi Email Spam Menggunakan Teorema Bayes
### **Mata Kuliah: Statistika dan Probabilitas (Statpro)**
---

## 📚 Deskripsi Proyek
Spam filtering (penyaringan pesan sampah/email penipuan) merupakan salah satu aplikasi paling klasik dan sukses dari **Teorema Bayes (Bayes' Theorem)** dalam bidang *Natural Language Processing (NLP)* dan *Machine Learning*.

Dalam proyek ini, kita akan:
1. Melakukan **Exploratory Data Analysis (EDA)** dan analisis statistik deskriptif terhadap dataset pesan spam dan ham (bukan spam).
2. Menerapkan pembersihan teks (*Text Preprocessing*) dan ekstraksi fitur menggunakan **TF-IDF (Term Frequency - Inverse Document Frequency)**.
3. Membangun model klasifikasi berbasis probabilitas:
   * **Multinomial Naïve Bayes** (Penerapan langsung Teorema Bayes)
   * **Complement Naïve Bayes** (Varian Bayes khusus untuk menangani *imbalanced dataset*)
   * **Logistic Regression** (Sebagai model pembanding berbasis *log-odds*)
4. Mengevaluasi performa model menggunakan metrik statistik: **Precision, Recall, F1-Score, Confusion Matrix**, dan **ROC-AUC**.
5. Menyediakan antarmuka uji coba interaktif untuk menguji pesan kustom secara langsung.

--- 
## 1. Setup Environment & Import Library
Kita mengimpor pustaka utama untuk manipulasi data (`pandas`, `numpy`), visualisasi (`matplotlib`, `seaborn`), pengolahan bahasa alami (`nltk`), dan pemodelan statistik (`scikit-learn`).

In [ ]:
# Import pustaka standar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string

# Pustaka NLP (Natural Language Toolkit)
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Scikit-Learn untuk Feature Extraction & Modeling
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Download resource NLTK yang dibutuhkan
nltk.download('stopwords')
nltk.download('punkt')

# Konfigurasi tampilan plot
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
print("✅ Seluruh library berhasil diimpor!")

--- 
## 2. Pemuatan Dataset (Data Loading)

> **Catatan untuk Google Colab:**
> Jika Anda menjalankan notebook ini di Google Colab, unggah file `spam.csv` melalui panel files di sebelah kiri, atau mount Google Drive Anda.

In [ ]:
# Load file dataset spam.csv
import os

file_path = 'Dataset/spam.csv' if os.path.exists('Dataset/spam.csv') else 'spam.csv'

# Membaca dataset dengan pandas
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='latin-1')

# Menghapus kolom index yang tidak bernama jika ada
if 'Unnamed: 0' in df.columns or df.columns[0] == '':
    df = df.iloc[:, 1:]

# Pastikan nama kolom standar
df.columns = ['label', 'message']

print(f"Ukuran dataset awal: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

In [ ]:
# Pemeriksaan Data: Missing Value & Data Duplikat
print("=== Missing Values ===")
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f"\nJumlah data duplikat: {duplicate_count}")

# Menghapus duplikat agar model tidak bias
if duplicate_count > 0:
    df = df.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"Data setelah penghapusan duplikat: {len(df)} baris")

# Encode Label: 'ham' -> 0, 'spam' -> 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

--- 
## 3. Exploratory Data Analysis (EDA) & Statistik Deskriptif

Dalam statistika, memahami sebaran data dan karakteristik populasi adalah langkah awal yang fundamental sebelum melakukan pemodelan probabilitas.

In [ ]:
# 1. Distribusi Kelas (Class Imbalance Analysis)
class_counts = df['label'].value_counts()
percentages = df['label'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], palette=['#2b5c8f', '#d9534f'])
axes[0].set_title("Frekuensi Pesan: Ham vs Spam", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Jumlah Data")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 40, f"{v} ({percentages.iloc[i]:.1f}%)", ha='center', fontweight='bold')

# Donut chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', startangle=90,
           colors=['#2b5c8f', '#d9534f'], wedgeprops=dict(width=0.4, edgecolor='white'))
axes[1].set_title("Proporsi Kelas Pesan", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Interpretasi Statistik:")
print(f"- Data bersifat IMBALANCED dengan mayoritas Ham ({percentages['ham']:.2f}%) dan minoritas Spam ({percentages['spam']:.2f}%).")
print("- Hal ini menandakan akurasi saja tidak cukup; kita wajib menganalisis Precision, Recall, dan F1-Score.")

In [ ]:
# 2. Analisis Statistik Karakteristik Panjang Pesan
df['char_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

# Ringkasan statistik deskriptif
summary_stats = df.groupby('label')[['char_length', 'word_count']].agg(['mean', 'median', 'std', 'min', 'max'])
print("Ringkasan Statistik Deskriptif (Karakter & Kata):")
display(summary_stats)

# Visualisasi Distribusi Panjang Karakter
plt.figure(figsize=(12, 5))
sns.histplot(df[df['label'] == 'ham']['char_length'], color='#2b5c8f', label='Ham', kde=True, bins=50, alpha=0.6)
sns.histplot(df[df['label'] == 'spam']['char_length'], color='#d9534f', label='Spam', kde=True, bins=50, alpha=0.6)
plt.title("Distribusi Panjang Karakter: Ham vs Spam", fontsize=14, fontweight='bold')
plt.xlabel("Jumlah Karakter per Pesan")
plt.ylabel("Frekuensi")
plt.xlim(0, 300)
plt.legend()
plt.show()

### 💡 Temuan Statistik Deskriptif:
* Pesan **Spam** memiliki rata-rata panjang karakter yang jauh lebih tinggi (~130-160 karakter) dengan variansi lebih kecil, karena umumnya disusun padat untuk memuat penawaran atau nomor kontak.
* Pesan **Ham** memiliki panjang karakter yang bervariasi (rata-rata ~70 karakter), cenderung pendek dalam percakapan sehari-hari.

--- 
## 4. Text Preprocessing (Pembersihan Teks)

Tahap preprocessing bertujuan untuk mereduksi noise dan menyederhanakan ruang fitur (*vocabulary*) sehingga model statistik dapat mengestimasi probabilitas kata dengan lebih akurat:
1. **Case Folding:** Mengubah semua huruf menjadi huruf kecil (*lowercase*).
2. **Regex Cleaning:** Menghapus URL, angka, tanda baca, dan karakter non-alfanumerik.
3. **Stopwords Removal:** Menghapus kata umum bahasa Inggris (seperti *is, the, at, on*) yang tidak memiliki muatan informasi pembeda.
4. **Stemming:** Mengembalikan kata ke bentuk dasarnya menggunakan *Porter Stemmer* (misal: *winning*, *wins* → *win*).

In [ ]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Hapus URL / Link web
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 3. Hapus tag HTML / karakter khusus
    text = re.sub(r'<.*?>', '', text)
    # 4. Hapus angka dan tanda baca
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 5. Tokenisasi kata
    words = text.split()
    # 6. Hapus stopwords & lakukan stemming
    cleaned_words = [ps.stem(w) for w in words if w not in stop_words and len(w) > 1]
    return ' '.join(cleaned_words)

# Terapkan fungsi preprocessing ke seluruh pesan
df['clean_message'] = df['message'].apply(preprocess_text)

# Tampilkan perbandingan sebelum dan sesudah
df[['message', 'clean_message']].head(5)

--- 
## 5. Feature Extraction: TF-IDF (Term Frequency - Inverse Document Frequency)

Dalam statistika teks, kita mengubah dokumen teks menjadi representasi numerik. **TF-IDF** mengukur seberapa penting sebuah kata dalam dokumen relatif terhadap seluruh korpus:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

Di mana:
* $\text{TF}(t, d)$: Frekuensi kemunculan term $t$ dalam pesan $d$.
* $\text{IDF}(t, D) = \log \left( \frac{1 + |D|}{1 + |\{d \in D : t \in d\}|} \right) + 1$: Memberikan bobot lebih kecil pada kata yang muncul di hampir semua pesan, dan bobot lebih besar pada kata pembeda yang unik.

In [ ]:
# Inisialisasi TF-IDF Vectorizer
# Kita batasi fitur ke 2.500 kata paling informatif dengan n-gram (unigram dan bigram)
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))

X = tfidf.fit_transform(df['clean_message']).toarray()
y = df['label_num'].values

print(f"Bentuk Matriks Fitur X: {X.shape[0]} sampel, {X.shape[1]} fitur kata/n-gram")
print(f"Bentuk Vektor Target y : {y.shape[0]} label")

--- 
## 6. Pembagian Data (Train-Test Split) dengan Stratifikasi

> **Prinsip Statistika:**
> Karena proporsi kelas tidak seimbang (~87% ham, ~13% spam), kita wajib menggunakan parameter `stratify=y`. Hal ini menjamin rasio kelas pada data latih (*training set*) dan data uji (*testing set*) sama persis dengan populasi aslinya.

In [ ]:
# Split 80% data latih dan 20% data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Jumlah Data Latih (Training): {X_train.shape[0]} sampel")
print(f"Jumlah Data Uji   (Testing) : {X_test.shape[0]} sampel")
print(f"Proporsi Spam pada Train: {(y_train.sum() / len(y_train))*100:.2f}%")
print(f"Proporsi Spam pada Test : {(y_test.sum() / len(y_test))*100:.2f}%")

--- 
## 7. Pemodelan Berbasis Probabilitas (Statistical Modeling)

### Landasan Teori: Teorema Bayes
Probabilitas sebuah pesan adalah **Spam** jika diketahui kumpulan kata-kata $\mathbf{x} = (x_1, x_2, \dots, x_n)$ dirumuskan oleh Teorema Bayes sebagai berikut:

$$P(\text{Spam} \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid \text{Spam}) \cdot P(\text{Spam})}{P(\mathbf{x})}$$

Asumsi *"Naïve"* mengasumsikan bahwa kemunculan setiap kata saling bebas secara kondisional (*conditionally independent*):
$$P(\mathbf{x} \mid \text{Spam}) = \prod_{i=1}^n P(x_i \mid \text{Spam})$$

Kita membandingkan tiga model statistik:
1. **Multinomial Naïve Bayes (MNB):** Model standar berbasis frekuensi kemunculan kata dengan *Laplace smoothing* ($\alpha = 1.0$).
2. **Complement Naïve Bayes (CNB):** Mengestimasi parameter dari *komplemen* setiap kelas, sangat efektif untuk menangani distribusi kelas yang *imbalanced*.
3. **Logistic Regression (LR):** Model probabilistik pembanding berbasis estimasi parameter *Maximum Likelihood* melalui fungsi logit.

In [ ]:
# Inisialisasi Model
models = {
    "Multinomial Naïve Bayes": MultinomialNB(alpha=1.0),
    "Complement Naïve Bayes": ComplementNB(alpha=1.0),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

trained_models = {}
predictions = {}
pred_probabilities = {}

for name, model in models.items():
    # Melatih model pada data latih
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Prediksi label kelas dan probabilitas pada data uji
    predictions[name] = model.predict(X_test)
    pred_probabilities[name] = model.predict_proba(X_test)[:, 1]
    
    print(f"✅ {name} berhasil dilatih!")

--- 
## 8. Evaluasi Performa Model & Analisis Statistik

Dalam evaluasi deteksi spam:
* **False Positive (FP):** Pesan penting (Ham) keliru diprediksi sebagai Spam → **Sangat berbahaya** (pengguna kehilangan email penting).
* **False Negative (FN):** Pesan Spam lolos ke kotak masuk → Mengganggu, namun risiko lebih rendah dibanding FP.
* Oleh karena itu, metrik **Precision** (meminimalkan FP) dan **F1-Score** menjadi perhatian utama.

In [ ]:
# 1. Ringkasan Metrik Evaluasi
eval_results = []

for name in models.keys():
    y_pred = predictions[name]
    y_prob = pred_probabilities[name]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    eval_results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision (Spam)': prec,
        'Recall (Spam)': rec,
        'F1-Score (Spam)': f1,
        'ROC-AUC': roc_auc
    })

df_eval = pd.DataFrame(eval_results)
print("Tabel Evaluasi Komparatif Model:")
display(df_eval.style.highlight_max(axis=0, subset=['Accuracy', 'Precision (Spam)', 'Recall (Spam)', 'F1-Score (Spam)', 'ROC-AUC'], color='#d1e7dd'))

In [ ]:
# 2. Visualisasi Confusion Matrix untuk Setiap Model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Pred Ham', 'Pred Spam'], yticklabels=['Actual Ham', 'Actual Spam'])
    axes[i].set_title(f"Confusion Matrix:\n{name}", fontsize=12, fontweight='bold')
    axes[i].set_ylabel("Actual Label")
    axes[i].set_xlabel("Predicted Label")

plt.tight_layout()
plt.show()

In [ ]:
# 3. Visualisasi Kurva ROC (Receiver Operating Characteristic) & AUC
plt.figure(figsize=(9, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for (name, y_prob), color in zip(pred_probabilities.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {roc_auc:.4f})")

plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Chance (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=11)
plt.ylabel('True Positive Rate (TPR / Recall)', fontsize=11)
plt.title('Perbandingan Kurva ROC (Receiver Operating Characteristic)', fontsize=13, fontweight='bold')
plt.legend(loc="lower right")
plt.show()

--- 
## 9. Pengujian Pesan Interaktif (Interactive Inference)

Berikut adalah fungsi untuk menguji teks email/SMS baru secara langsung. Model akan mengeluarkan prediksi kelas beserta tingkat probabilitas posterior $P(\text{Spam} \mid \text{Teks})$.

In [ ]:
def classify_email(message_text, model=trained_models['Multinomial Naïve Bayes'], vectorizer=tfidf):
    # 1. Preprocessing teks
    cleaned = preprocess_text(message_text)
    
    # 2. Vektorisasi TF-IDF
    vec = vectorizer.transform([cleaned]).toarray()
    
    # 3. Prediksi dan Probabilitas Posterior
    prediction = model.predict(vec)[0]
    probabilities = model.predict_proba(vec)[0]
    
    prob_ham = probabilities[0] * 100
    prob_spam = probabilities[1] * 100
    
    label_str = "🚨 SPAM" if prediction == 1 else "✅ HAM (Bukan Spam)"
    
    print(f"\n{'='*55}")
    print(f"Pesan Input   : {message_text}")
    print(f"Hasil Prediksi: {label_str}")
    print(f"Probabilitas  : Ham = {prob_ham:.2f}% | Spam = {prob_spam:.2f}%")
    print(f"{'='*55}")

# Contoh Uji Coba Beberapa Pesan Simulasi:
test_cases = [
    "Hey, are we still meeting for lunch at 12 at the faculty cafeteria?",
    "CONGRATULATIONS! You have won a $1,000 Walmart Gift Card. Click here to claim your prize immediately!",
    "Dear student, please find attached the lecture notes and assignment guidelines for Statistics.",
    "URGENT: Your mobile account has been credited with £500 bonus. Call 087123456 now to verify code 4492!"
]

for case in test_cases:
    classify_email(case)

In [ ]:
# Silakan ketik dan uji pesan Anda sendiri di bawah ini:
custom_message = "URGENT! You have been selected for a free holiday trip to Bali. Call us to claim!"
classify_email(custom_message)

--- 
## 10. Kesimpulan & Pembahasan Statistika

1. **Keunggulan Teorema Bayes (Multinomial Naïve Bayes):**
   * Meskipun memiliki asumsi penyederhanaan yang kuat (*naïve conditional independence*), Naïve Bayes terbukti sangat efektif dan cepat dalam mengklasifikasikan dokumen teks.
   * MNB mampu memberikan **Precision** yang sangat tinggi pada kelas spam, yang berarti risiko menandai email penting sebagai spam (*False Positive*) sangat minim.

2. **Pentingnya Penanganan Imbalanced Data:**
   * Proporsi kelas yang tidak seimbang (~87% vs ~13%) memerlukan teknik seperti *stratified sampling* dan metrik evaluasi yang tepat (*F1-Score*, *ROC-AUC*, *Precision-Recall*).
   * Model varian **Complement Naïve Bayes (CNB)** memberikan keseimbangan *recall* yang lebih baik jika targetnya adalah mendeteksi sebanyak mungkin spam yang masuk.

3. **Peran TF-IDF:**
   * Bobot TF-IDF secara statistik berhasil menekan pengaruh kata-kata umum yang sering muncul di semua dokumen dan memperkuat bobot probabilitas kata-kata kunci pembeda spam (seperti *'free', 'win', 'prize', 'claim', 'call'*).